In [ ]:
import pandas as pd
import numpy as np
import os
os.listdir()

In [ ]:
import re

def normalize_text(text):
    if not isinstance(text, str):
        return ""

    # remove surrounding quotes
    text = text.strip()
    text = text.strip('"').strip("'")

    # normalize spaces
    text = re.sub(r"\s+", " ", text)

    return text

In [ ]:
def is_valid_pair(src, tgt):
    if not src or not tgt:
        return False

    # remove extremely short samples
    if len(src.split()) < 2 or len(tgt.split()) < 2:
        return False

    # remove obvious junk symbols
    junk_patterns = ["x-men", "bitcoin", "blockchain"] # these patterns were so prominent at the dataset
    if any(p in src.lower() or p in tgt.lower() for p in junk_patterns):
        return False

    return True

In [ ]:
def clean_dataset(data):
    cleaned = []

    for item in data:
        src = normalize_text(item.get("source", ""))
        tgt = normalize_text(item.get("target", ""))

        if not is_valid_pair(src, tgt):
            continue

        cleaned.append({
            "source": src,
            "target": tgt,
            "score": item.get("score", 0)
        })

    return cleaned

In [ ]:
clean_base = clean_dataset(base_data)

print(len(base_data), "→", len(clean_base))

In [ ]:
import re

def looks_english(text):
    return bool(re.search(r"[a-zA-Z]", text))

def filter_language(data):
    return [
        x for x in data
        if looks_english(x["source"]) and looks_english(x["target"])
    ]

In [ ]:
def add_lang_tags_en_wa(item):
    return {
        "source": item["source"],
        "target": item["target"],
        "src_lang": "eng_Latn",
        "tgt_lang": "war_Latn"
    }

In [ ]:
clean_base = [add_lang_tags_en_wa(x) for x in clean_base]
gold_data = [add_lang_tags_en_wa(x) for x in gold_data]

In [ ]:
def invert(item):
    return {
        "source": item["target"],
        "target": item["source"],
        "src_lang": "war_Latn",
        "tgt_lang": "eng_Latn"
    }

base_rev = [invert(x) for x in clean_base]
gold_rev = [invert(x) for x in gold_data]

In [ ]:
merged_data = clean_base + gold_data

In [ ]:
save_jsonl(merged_data, os.path.join(save_dir, "merged_raw.jsonl"))

Another Process of Cleaning

In [ ]:
import re

def normalize_text(text):
    text = text.strip()
    text = text.replace('\\"', '"')
    text = re.sub(r"\s+", " ", text)
    return text

In [ ]:
def is_clean_pair(item):
    src = normalize_text(item["source"])
    tgt = normalize_text(item["target"])

    # Rule 1: must not be empty
    if not src or not tgt:
        return False

    # Rule 2: too short or broken fragments
    if len(src.split()) < 2 or len(tgt.split()) < 2:
        return False

    # Rule 3: reject if too many non-letters (noise / gibberish)
    def alpha_ratio(text):
        letters = sum(c.isalpha() for c in text)
        return letters / max(len(text), 1)

    if alpha_ratio(src) < 0.6 or alpha_ratio(tgt) < 0.6:
        return False

    # Rule 4: reject heavy code-switching in target (non-Waray noise)
    bad_tokens = ["gue", "iya", "sama", "john"]
    if any(b in tgt.lower() for b in bad_tokens):
        return False

    # Rule 5: remove mismatched conversational junk
    if tgt.lower() in ["haha", "lol", "ok", "yes"]:
        return False

    return True

In [ ]:
cleaned_data = []

for item in merged_data:
    item["source"] = normalize_text(item["source"])
    item["target"] = normalize_text(item["target"])

    if is_clean_pair(item):
        cleaned_data.append(item)

print("Before:", len(merged_data))
print("After:", len(cleaned_data))

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd

wordlist_path = "waray-wordlist.csv"
word_df = pd.read_csv(wordlist_path)

In [ ]:
filtered_word_df = word_df[word_df["Count"] >= 3]

In [ ]:
waray_vocab = set(filtered_word_df["Word"].astype(str).str.lower())

In [ ]:
def contains_waray_words(text, min_hits=1):
    text = text.lower().split()
    hits = sum(1 for w in text if w in waray_vocab)
    return hits >= min_hits

In [ ]:
import re

def normalize(text):
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    text = text.replace('\\"', '"')
    return text

def is_clean_pair(item):
    src = normalize(item["source"])
    tgt = normalize(item["target"])

    # basic validity
    if len(src) < 3 or len(tgt) < 3:
        return False

    # alpha ratio filter
    def alpha_ratio(t):
        return sum(c.isalpha() for c in t) / max(len(t), 1)

    if alpha_ratio(src) < 0.6 or alpha_ratio(tgt) < 0.6:
        return False

    # Waray wordlist filter
    if item.get("tgt_lang") == "war_Latn":
        if not contains_waray_words(tgt, min_hits=1):
            return False

    return True

In [ ]:
cleaned_data = [
    {
        **item,
        "source": normalize(item["source"]),
        "target": normalize(item["target"])
    }
    for item in merged_data
    if is_clean_pair(item)
]

In [ ]:
from itertools import tee

def ngrams(words, n):
    iters = tee(words, n)
    for i, it in enumerate(iters):
        for _ in range(i):
            next(it, None)
    return zip(*iters)

def waray_phrase_score(text):
    words = text.lower().split()

    # unigram hits
    unigram_hits = sum(1 for w in words if w in waray_vocab)

    # phrase hits (bigrams)
    bigrams = [" ".join(bg) for bg in ngrams(words, 2)]
    bigram_hits = sum(1 for bg in bigrams if bg in waray_vocab)

    return unigram_hits + bigram_hits

In [ ]:
from difflib import SequenceMatcher

def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

def is_bad_similarity(src, tgt):
    sim = similarity(src, tgt)

    # too similar = likely bad translation/copy noise
    if sim > 0.85:
        return True

    # too unrelated = weak alignment
    if sim < 0.10:
        return True

    return False

def is_pure_waray(text):
    text = text.lower()

    # reject obvious non-Waray languages
    bad_markers = [
        "the", "and", "you", "i'm", "we are",   # English leakage
        "gue", "sama", "aku", "kita kan"        # Indonesian/Tagalog mix
    ]

    if any(b in text for b in bad_markers):
        return False

    # must contain at least some Waray signal
    return waray_phrase_score(text) >= 1

def deduplicate(data):
    seen = set()
    unique_data = []

    for item in data:
        src = normalize(item["source"]).lower()
        tgt = normalize(item["target"]).lower()

        key = (src, tgt)

        if key not in seen:
            seen.add(key)
            unique_data.append(item)

    return unique_data

def is_valid_pair(item):
    src = normalize(item["source"])
    tgt = normalize(item["target"])

    # enforce language direction
    if item["src_lang"] != "eng_Latn" or item["tgt_lang"] != "war_Latn":
        return False

    # basic length check
    if len(src.split()) < 2 or len(tgt.split()) < 2:
        return False

    # similarity filter
    if is_bad_similarity(src, tgt):
        return False

    # English purity check
    if not any(c.isalpha() for c in src):
        return False

    # Waray purity + lexicon + phrase detection
    if not is_pure_waray(tgt):
        return False

    return True



In [ ]:
cleaned_data = []

for item in merged_data:
    item["source"] = normalize(item["source"])
    item["target"] = normalize(item["target"])

    if is_valid_pair(item):
        cleaned_data.append(item)

print("Before:", len(merged_data))
print("After filtering:", len(cleaned_data))

In [ ]:
cleaned_data = deduplicate(cleaned_data)

print("After dedup:", len(cleaned_data))

In [ ]:
def invert(item):
    return {
        "source": item["target"],
        "target": item["source"],
        "src_lang": "war_Latn",
        "tgt_lang": "eng_Latn"
    }

inverted_data = [invert(x) for x in cleaned_data]

In [ ]:
weighted_clean = []
weighted_inverted = []

for item in cleaned_data:
    item["weight"] = 1.0
    weighted_clean.append(item)

for item in inverted_data:
    item["weight"] = 0.5
    weighted_inverted.append(item)

In [ ]:
final_dataset = weighted_clean + weighted_inverted

In [ ]:
import random

random.shuffle(final_dataset)

In [ ]:
from fasttext import load_model

lid_model = load_model("lid.176.ftz")

Another set of cleaning

In [ ]:
import json
import pandas as pd
from fasttext import load_model

In [ ]:
lid_model = load_model("lid.176.ftz")

In [ ]:
def load_jsonl(path):
    data=[]
    with open(path,"r",encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    return data

merged_data = load_jsonl(
    "path_to_data"
)

In [ ]:
# 5. Reloading of Waray wordlist
word_df = pd.read_csv("link to csv")
word_df = word_df[word_df["Count"] >= 3]

waray_vocab = set(
    word_df["Word"].astype(str).str.lower()
)

In [ ]:
def is_english(text):
    pred = lid_model.predict(text.replace("\n", " "))[0][0]
    return pred == "__label__en"

In [ ]:
import re

bad_languages = {
    "__label__tl",  # Tagalog
    "__label__id",  # Indonesian
}

def is_waray(text):

    text = normalize(text.lower())

    words = re.findall(r"\b\w+\b", text)

    if len(words) == 0:
        return False

    # Waray vocabulary matches
    waray_hits = sum(
        1 for w in words
        if w in waray_vocab
    )

    # obvious foreign language markers
    foreign_markers = {
        "alguien","visto",
        "gue","gw","ngomong",
        "productes","bonjour",
        "gracias","hola"
    }

    foreign_hits = sum(
        1 for w in words
        if w in foreign_markers
    )

    waray_ratio = waray_hits / len(words)

    # require minimum Waray density
    if waray_ratio < 0.20:
        return False

    # reject if foreign words exist
    if foreign_hits > 0:
        return False

    return True

In [ ]:
import re

def excessive_source_overlap(src, tgt):

    src_words = set(re.findall(r"\b\w+\b", src.lower()))
    tgt_words = set(re.findall(r"\b\w+\b", tgt.lower()))

    # common words to ignore
    ignore = {
        "new","york","stock","exchange",
        "forex","trading",
        "tommy","hilfiger",
        "papua","guinea"
    }

    src_words -= ignore
    tgt_words -= ignore

    overlap = len(src_words & tgt_words)

    ratio = overlap / max(len(tgt_words),1)

    return ratio > 0.60

    print("Overlap ratio:", ratio)
    print("Shared:", src_words & tgt_words)

In [ ]:
def validate_pair(item):

    src=normalize(item["source"])
    tgt=normalize(item["target"])

    if item["src_lang"]!="eng_Latn":
        return False,"wrong source label"

    if item["tgt_lang"]!="war_Latn":
        return False,"wrong target label"

    if not is_english(src):
        return False,"source not English"

    if not is_waray(tgt):
        return False,"target not Waray"

    if excessive_source_overlap(src,tgt):
        return False,"target copied source"

    return True,"valid"

In [ ]:
from difflib import SequenceMatcher

def semantic_sanity(src,tgt):

    src=src.lower()
    tgt=tgt.lower()

    similarity=SequenceMatcher(
        None,
        src,
        tgt
    ).ratio()

    return similarity<0.90

    if not semantic_sanity(src,tgt):
      return False,"too similar"

In [ ]:
template_noise = [
    "there are",
    "products",
    "lyrics",
    "mp3",
    "official website",
    "open directory project"
]

def is_template_noise(text):

    text=text.lower()

    return any(
        x in text
        for x in template_noise
    )

    if is_template_noise(src):
      return False,"template noise"

    if is_template_noise(tgt):
      return False,"template noise"

In [ ]:
import re

def normalize(text):
    text = str(text).strip()

    # fix escaped quotes
    text = text.replace('\\"', '"')

    # remove repeated spaces
    text = re.sub(r"\s+", " ", text)

    # normalize whitespace/newlines
    text = text.replace("\n", " ")

    return text

In [ ]:
validated_data = []
rejected_data = []

for item in merged_data:

    valid, reason = validate_pair(item)

    if valid:
        validated_data.append(item)
    else:
        rejected_data.append({
            "item": item,
            "reason": reason
        })


print("="*80)
print("DATASET SUMMARY")
print("="*80)

print("Before:", len(merged_data))
print("Validated:", len(validated_data))
print("Rejected:", len(rejected_data))


print("\n===== VALID SAMPLES =====")

for sample in validated_data[:5]:
    print("SOURCE:", sample["source"])
    print("TARGET:", sample["target"])
    print("SRC_LANG:", sample["src_lang"])
    print("TGT_LANG:", sample["tgt_lang"])
    print("-"*80)


print("\n===== REJECTED SAMPLES =====")

for sample in rejected_data[:10]:

    item = sample["item"]

    print("REASON:", sample["reason"])
    print("SOURCE:", item["source"])
    print("TARGET:", item["target"])
    print("SRC_LANG:", item["src_lang"])
    print("TGT_LANG:", item["tgt_lang"])
    print("-"*80)

In [ ]:
import json

def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

In [ ]:
def invert(item):
    return {
        "source": item["target"],
        "target": item["source"],
        "src_lang": "war_Latn",
        "tgt_lang": "eng_Latn"
    }

In [ ]:
waray_reverse = [invert(x) for x in validated_data]

In [ ]:
final_data = validated_data + waray_reverse

In [ ]:
en_to_war = []
war_to_en = []

for item in validated_data:

    # ORIGINAL: English → Waray
    en_item = item.copy()
    en_item["weight"] = 1.0
    en_to_war.append(en_item)

    # INVERTED: Waray → English
    inv_item = invert(item)
    inv_item["weight"] = 0.5
    war_to_en.append(inv_item)

In [ ]:
final_dataset = en_to_war + war_to_en

print("EN→WAR samples:", len(en_to_war))
print("WAR→EN samples:", len(war_to_en))
print("TOTAL:", len(final_dataset))

In [ ]:
import random

random.shuffle(final_dataset)